# Camada Silver — Transformação e Limpeza

## Objetivo
Transformar as 5 tabelas Bronze em 7 tabelas Silver limpas, tipadas e em português,
prontas para modelagem dimensional na camada Gold.

**Convenções deste projeto:**
- Comentários e textos: português (PT-BR)
- Identificadores (variáveis, funções): inglês (EN)
  - Constantes: `UPPER_SNAKE_CASE`
  - Demais variáveis: `lower_snake_case`
- Colunas e nomes de tabela: português, `snake_case`
- Símbolos nos logs: ✅ ⚠️ ❌

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime

# Padronização dos nomes dos schemas e volumes
CATALOG = "movies_medallion"
BRONZE_SCHEMA_NAME = "bronze"
SILVER_SCHEMA_NAME = "silver"
GOLD_SCHEMA_NAME = "gold"

BRONZE_SCHEMA = f"{CATALOG}.{BRONZE_SCHEMA_NAME}"
SILVER_SCHEMA = f"{CATALOG}.{SILVER_SCHEMA_NAME}"
GOLD_SCHEMA = f"{CATALOG}.{GOLD_SCHEMA_NAME}"

# Símbolos padronizados de log
OK = "✅"
FAIL = "❌"
WARN = "⚠️"

# Para verificação visual
print(f"CATALOG: {CATALOG}")
print(f"BRONZE_SCHEMA: {BRONZE_SCHEMA}")
print(f"SILVER_SCHEMA: {SILVER_SCHEMA}")
print(f"GOLD_SCHEMA: {GOLD_SCHEMA}")

CATALOG: movies_medallion
BRONZE_SCHEMA: movies_medallion.bronze
SILVER_SCHEMA: movies_medallion.silver
GOLD_SCHEMA: movies_medallion.gold


## silver.tb_info_filmes
Metadados principais dos filmes.
- Normalização e tradução de status
- Dedup por versão mais recente (ingestion_datetime)
- Conversão de data multi-formato com coalesce
- Derivada: ano_lancamento

In [0]:
# O dicionário para facilitar a tradução das colunas e eventual manutenção.
col_names_translated = {
    "id": "id_filme",
    "title": "titulo",
    "original_title": "titulo_original",
    "release_date": "data_lancamento",
    "runtime": "duracao_minutos",
    "original_language": "idioma_original",
    "status": "status_filme",
    "overview": "sinopse",
    "tagline": "frase_divulgacao",
    "ingestion_datetime": "ingestao_datetime",
}

# Window para pegar o resultado mais recentemente ingerido de id_filme
w = Window.partitionBy("id_filme").orderBy(
    F.col("ingestao_datetime").desc(), F.col("id_filme").asc()
)

df_bronze_movies_info = spark.table(f"{BRONZE_SCHEMA}.tb_movies_info")

# Investiga e retorna quais os formatos possíveis da coluna release_date para posterior padronização
# Troca dígitos por "9", letras por "A" e agrupa
display(
    df_bronze_movies_info.select("release_date")
    .withColumn(
        "assinatura",
        F.regexp_replace(
            F.regexp_replace(F.trim(F.lower(F.col("release_date"))), r"\d", "9"),
            r"[a-z]",
            "A",
        ),
    )
    .groupBy("assinatura")
    .count()
    .orderBy(F.col("count").desc())
)

# Padronizações e transformações para criação da tabela silver
df_silver_movies_info = (
    # Renomeação com base no dicionário
    df_bronze_movies_info.select(list(col_names_translated.keys()))
    .withColumnsRenamed(col_names_translated)
    .withColumn("titulo", F.initcap(F.col("titulo")))
    # Normalização da coluna status_filme e tradução de seus status
    .withColumn(
        "status_filme",
        F.trim(F.lower(F.regexp_replace("status_filme", r"[-\s]+", " "))),
    )
    .withColumn(
        "status_filme",
        F.when(F.col("status_filme").isNull(), "Não Informado")
        .when(F.col("status_filme") == "released", "Lançado")
        .when(F.col("status_filme") == "post production", "Pós-Produção")
        .when(F.col("status_filme") == "in production", "Em Produção")
        .when(F.col("status_filme") == "planned", "Planejado")
        .when(F.col("status_filme") == "rumored", "Rumores")
        .when(F.col("status_filme") == "canceled", "Cancelado")
        .otherwise("Não Informado")
    )
    .withColumn("duracao_minutos", F.col("duracao_minutos").try_cast("int"))
    # Usa a window criada para selecionar apenas o registro mais recente por id_filme (dedupação)
    .withColumn("rk", F.row_number().over(w))
    .filter(F.col("rk") == 1)
    .drop("rk")
    # Testa e padroniza os 3 formatos de datas identificados na inspeção
    .withColumn(
        "data_lancamento",
        F.coalesce(
            F.try_to_date(F.col("data_lancamento"), "yyyy-MM-dd"),
            F.try_to_date(F.col("data_lancamento"), "dd/MM/yyyy"),
            F.try_to_date(F.col("data_lancamento"), "dd-MM-yyyy"),
        ),
    )
    .withColumn("ano_lancamento", F.year(F.col("data_lancamento")))
    .drop("ingestao_datetime")
)

# Validações
bronze_total = df_bronze_movies_info.count()
bronze_unique = df_bronze_movies_info.select("id").distinct().count()
silver_total = df_silver_movies_info.count()
silver_unique = df_silver_movies_info.select("id_filme").distinct().count()

print(f"Bronze total:      {bronze_total}")
print(f"Bronze ids únicos: {bronze_unique}")
print(f"Silver total:      {silver_total}")
print(f"Silver ids únicos: {silver_unique}")



# Verifica se algum id_filme é nulo
nulos_id = df_silver_movies_info.filter(F.col("id_filme").isNull()).count()
if nulos_id > 0:
    print(f"{FAIL} {nulos_id} registros com id_filme nulo")
    raise Exception(f"id_filme nulo em {nulos_id} registros — chave natural não pode ser NULL.")

# Silver deve ter exatamente 1 linha por id_filme
if silver_total != silver_unique:
    print(f"{FAIL} Dedup falhou: {silver_total} linhas vs {silver_unique} id_filme únicos. Investigar window function (rk)")
    raise Exception(
        f"Dedup por id_filme falhou: {silver_total} linhas, "
        f"mas apenas {silver_unique} id_filme únicos. "
        f"Investigar window function (rk)."
    )

# Silver deve ter o mesmo numero de filmes que bronze
if silver_total != bronze_unique:
    print(f"{FAIL} Perda de filmes: bronze tinha {bronze_unique} únicos, silver tem {silver_total}")
    raise Exception(
        f"Silver tem {silver_total} filmes, mas bronze tinha {bronze_unique} únicos. "
        f"Alguma linha foi perdida na transformação."
    )

print(f"{OK} Dedup OK e paridade com bronze mantida: {silver_total} filmes únicos")

display(
    df_silver_movies_info.select(
        F.count("*").alias("total"),
        F.sum(F.when(F.col("data_lancamento").isNull(), 1).otherwise(0)).alias("nulls"),
    )
)

display(
    df_silver_movies_info.groupBy("data_lancamento")
    .count()
    .orderBy("count", ascending=False)
    .limit(10)
)

# Estrutura: 10 colunas com nomes corretos
expected_cols = [
    "id_filme", "titulo", "titulo_original", "data_lancamento",
    "duracao_minutos", "idioma_original", "status_filme",
    "sinopse", "frase_divulgacao", "ano_lancamento",
]
assert df_silver_movies_info.columns == expected_cols, (
    f"Colunas erradas: esperado {expected_cols}, "
    f"obtido {df_silver_movies_info.columns}"
)
print(f"{OK} Estrutura OK — 10 colunas com nomes corretos")

df_silver_movies_info.printSchema()

# Persiste o DataFrame como tabela Delta no schema silver.
(df_silver_movies_info
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_SCHEMA}.tb_info_filmes"))

print(f"Tabela gravada: {SILVER_SCHEMA}.tb_info_filmes")

# Validação da tabela gravada
df_check = spark.table(f"{SILVER_SCHEMA}.tb_info_filmes")
print(f"Linhas na tabela: {df_check.count()}")
df_check.printSchema()
display(df_check.limit(5))

assinatura,count
9999-99-99,91541
99/99/9999,10602
99-99-9999,4717
null,62
AAAAA'A AAAAAAAA AAAAAAAAA. AAAAA'A AAAAAAAA AAAAAAA.,1
AAA AAAAAA AAAA AAAAAAAAA AAAAAAA,1
AAAAA AAAAAAAA AA AAAAAAAA AAA AAAA AA AAA AAAAAAA,1
AA AAAAA AAA A AAAAA AAAAAA,1
AA AAAAAAAAAAA AAAA AAAAA AAAAA AAAAA AAAAAAA AAAAAAAAAA AAAAAAAAA,1
AAA AAAA AAAAAA AAAAAAAAA AAA AAAAAAA AAAA,1


Bronze total:      106930
Bronze ids únicos: 97879
Silver total:      97879
Silver ids únicos: 97879
✅ Dedup OK e paridade com bronze mantida: 97879 filmes únicos


total,nulls
97879,2565


data_lancamento,count
null,2565
2016-01-01,469
2017-01-01,374
2018-01-01,284
2019-01-01,177
2020-01-01,173
2021-01-01,135
2020-10-01,128
2019-11-01,124
2019-10-18,118


✅ Estrutura OK — 10 colunas com nomes corretos
root
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- titulo_original: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = false)
 |-- sinopse: string (nullable = true)
 |-- frase_divulgacao: string (nullable = true)
 |-- ano_lancamento: integer (nullable = true)

Tabela gravada: movies_medallion.silver.tb_info_filmes
Linhas na tabela: 97879
root
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- titulo_original: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = true)
 |-- sinopse: string (nullable = true)
 |-- frase_divulgacao: string (nullable = true)
 |-- ano_lancamento: in

id_filme,titulo,titulo_original,data_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,ano_lancamento
1000004,Purple Beatz,Purple Beatz,2022-07-07,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.",A drum n bass romance.,2022
1000005,Aisha Brown: The First Black Woman Ever,Aisha Brown: The First Black Woman Ever,2020-02-14,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs.",null,2020
1000007,Kyle Brownrigg: Introducing Lyle,Kyle Brownrigg: Introducing Lyle,2022-05-27,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.",null,2022
1000011,Worth Your Weight In Gold,O Teu Peso Em Ouro,2022-07-14,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness.",null,2022
1000014,On Va Manquer !,On va manquer !,2018-05-15,0,fr,Lançado,null,null,2018


## silver.tb_financeiro_filmes
Métricas financeiras dos filmes em USD e BRL.
- Conversão segura de texto → decimal
- Conversão USD → BRL pela cotação mais recente
- Derivadas: lucro_usd, lucro_brl, margem_lucro_percentual

In [0]:
def normalize_monetary_value(column_name):
    """Limpa e converte valores monetários, tratando sufixos de magnitude (K/M)."""
    col = F.col(column_name)
    col_upper = F.upper(F.trim(col))

    # Detecta sufixo de magnitude (K = mil, M = milhão).
    # O \d antes do K/M garante que não casa com "N/A" ou texto solto.
    multiplier = (
        F.when(col_upper.rlike(r"\d\s*K$"), F.lit(1000))
        .when(col_upper.rlike(r"\d\s*M$"), F.lit(1000000))
        .otherwise(F.lit(1))
    )

    # Remove tudo que não é dígito, ponto ou hífen (símbolos, prefixos, sufixos)
    cleaned = F.regexp_replace(col, r"[^\d.-]", "")
    converted = cleaned.try_cast("decimal(18,2)") * multiplier

    # Valores zerados ou negativos viram ausentes
    return F.when(converted > 0, converted)


df_bronze_movies_financials = spark.table(f"{BRONZE_SCHEMA}.tb_movies_financials")

# Dicionário para facilitar a tradução das colunas e eventual manutenção.
col_names_translated = {
    "id": "id_filme",
    "budget": "orcamento_usd",
    "revenue": "receita_usd",
}

# Padronizações e transformações base
df_silver_movies_financials = (
    df_bronze_movies_financials.select(list(col_names_translated.keys()))
    .withColumnsRenamed(col_names_translated)
    .withColumn("orcamento_usd", normalize_monetary_value("orcamento_usd"))
    .withColumn("receita_usd", normalize_monetary_value("receita_usd"))
    .dropDuplicates(["id_filme"])
)

# Pega a cotação mais recente da tabela bronze
df_bronze_cotacao_dolar = spark.table(f"{BRONZE_SCHEMA}.tb_cotacao_dolar")

latest_row = (
    df_bronze_cotacao_dolar.orderBy(F.col("dataHoraCotacao").desc())
    .select("cotacaoCompra")
    .first()
)

# Verifica se a tabela bronze está vazia
if latest_row is None:
    print(f"{FAIL} Tabela de cotação vazia no bronze")
    raise Exception("Tabela de cotação vazia no bronze")

latest_rate = latest_row["cotacaoCompra"]
print(f"{OK} Cotação usada (última): {latest_rate}")

# Calcula valores monetários em BRL e o percentual de lucro
df_silver_movies_financials = (
    df_silver_movies_financials.withColumn(
        "orcamento_brl", F.col("orcamento_usd") * latest_rate
    )
    .withColumn("receita_brl", F.col("receita_usd") * latest_rate)
    .withColumn("lucro_usd", F.col("receita_usd") - F.col("orcamento_usd"))
    .withColumn("lucro_brl", F.col("receita_brl") - F.col("orcamento_brl"))
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            F.col("receita_usd").isNotNull() & (F.col("receita_usd") > 0),
            F.round((F.col("lucro_usd") / F.col("receita_usd")) * 100, 2),
        ),
    )
)

# Validações
expected_cols = [
    "id_filme", "orcamento_usd", "receita_usd", "orcamento_brl",
    "receita_brl", "lucro_usd", "lucro_brl", "margem_lucro_percentual",
]
assert df_silver_movies_financials.columns == expected_cols, (
    f"Colunas erradas: esperado {expected_cols}, "
    f"obtido {df_silver_movies_financials.columns}"
)
print(f"{OK} Estrutura OK — 8 colunas com nomes corretos")

# Verifica se algum id_filme é nulo
nulos_id = df_silver_movies_financials.filter(F.col("id_filme").isNull()).count()
if nulos_id > 0:
    print(f"{FAIL} {nulos_id} registros com id_filme nulo")
    raise Exception(f"id_filme nulo em {nulos_id} registros — chave natural não pode ser NULL.")

# Margem não pode passar de 100%
margem_invalida = df_silver_movies_financials.filter(
    F.col("margem_lucro_percentual") > 100
).count()
if margem_invalida > 0:
    print(f"{FAIL} {margem_invalida} filmes com margem > 100%")
    raise Exception(
        f"{margem_invalida} registros com margem_lucro_percentual > 100. "
        f"Indica inconsistência entre receita e orçamento (receita < orçamento não deveria "
        f"gerar margem > 100)."
    )

# A tabela não pode ter ficado totalmente vazia de valores monetários
nao_nulos_orc = df_silver_movies_financials.filter(
    F.col("orcamento_usd").isNotNull()
).count()
if nao_nulos_orc == 0:
    print(f"{FAIL} Nenhum orçamento válido. Investigar normalize_monetary_value.")
    raise Exception("Nenhum orçamento válido. Investigar normalize_monetary_value.")

print(f"{OK} Sem margens > 100%")
print(f"{OK} Orçamentos não-nulos: {nao_nulos_orc}")

# Visualização
display(
    df_silver_movies_financials.orderBy(F.col("receita_usd").desc_nulls_last())
    .select("id_filme", "orcamento_usd", "receita_usd")
    .limit(10)
)

display(
    df_silver_movies_financials.orderBy(F.col("receita_brl").desc_nulls_last()).limit(
        10
    )
)

# Persiste a tabela silver no catálogo
(
    df_silver_movies_financials.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_SCHEMA}.tb_financeiro_filmes")
)

print(f"{OK} Tabela gravada: {SILVER_SCHEMA}.tb_financeiro_filmes")

# Validação da tabela gravada
df_check = spark.table(f"{SILVER_SCHEMA}.tb_financeiro_filmes")
print(f"{OK} Linhas na tabela: {df_check.count()}")
df_check.printSchema()
display(df_check.limit(5))

✅ Cotação usada (última): 4.9692
✅ Estrutura OK — 8 colunas com nomes corretos
✅ Sem margens > 100%
✅ Orçamentos não-nulos: 8171


id_filme,orcamento_usd,receita_usd
299534,356000000.00,2800000000.00
76600,460000000.00,2320250281.00
299536,300000000.00,2052415039.00
634649,200000000.00,1921847111.00
420818,260000000.00,1663075401.00
361743,170000000.00,1488732821.00
346698,145000000.00,1428545028.00
502356,100000000.00,1355725263.00
284054,200000000.00,1349926083.00
181808,200000000.00,1332698830.00


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
299534,356000000.00,2800000000.00,1.7690352E9,1.391376E10,2444000000.00,1.21447248E10,87.29
76600,460000000.00,2320250281.00,2.285832E9,1.15297876963452E10,1860250281.00,9.2439556963452E9,80.17
299536,300000000.00,2052415039.00,1.49076E9,1.01988608117988E10,1752415039.00,8.7081008117988E9,85.38
634649,200000000.00,1921847111.00,9.9384E8,9.5500426639812E9,1721847111.00,8.556202663981199E9,89.59
420818,260000000.00,1663075401.00,1.291992E9,8.2641542826491995E9,1403075401.00,6.9721622826491995E9,84.37
361743,170000000.00,1488732821.00,8.44764E8,7.3978111341132E9,1318732821.00,6.5530471341132E9,88.58
346698,145000000.00,1428545028.00,7.20534E8,7.0987259531376E9,1283545028.00,6.3781919531376E9,89.85
502356,100000000.00,1355725263.00,4.9692E8,6.7368699768996E9,1255725263.00,6.2399499768996E9,92.62
284054,200000000.00,1349926083.00,9.9384E8,6.7080526916435995E9,1149926083.00,5.7142126916435995E9,85.18
181808,200000000.00,1332698830.00,9.9384E8,6.622447026036E9,1132698830.00,5.628607026036E9,84.99


✅ Tabela gravada: movies_medallion.silver.tb_financeiro_filmes
✅ Linhas na tabela: 99006
root
 |-- id_filme: string (nullable = true)
 |-- orcamento_usd: decimal(29,2) (nullable = true)
 |-- receita_usd: decimal(29,2) (nullable = true)
 |-- orcamento_brl: double (nullable = true)
 |-- receita_brl: double (nullable = true)
 |-- lucro_usd: decimal(30,2) (nullable = true)
 |-- lucro_brl: double (nullable = true)
 |-- margem_lucro_percentual: decimal(35,2) (nullable = true)



id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
284054,200000000.00,1349926083.00,9.9384E8,6.7080526916435995E9,1149926083.00,5.7142126916435995E9,85.18
447332,17000000.00,341000000.00,8.44764E7,1.6944972E9,324000000.00,1.6100208E9,95.01
337404,200000000.00,233503234.00,9.9384E8,1.1603242703927999E9,33503234.00,1.6648427039279985E8,14.35
512200,125000000.00,801693929.00,6.2115E8,3.9837774719867997E9,676693929.00,3.3626274719867997E9,84.41
348350,250000000.00,392952373.00,1.2423E9,1.9526589319115999E9,142952373.00,7.103589319115999E8,36.38


## silver.tb_metricas_engajamento

Métricas de popularidade, notas e votos (TMDB e IMDb).

- Limpeza de separadores decimais mistos (`,` e `.`)
- Conversão segura via `try_cast` (inválidos → NULL)
- Faixas de validação:
  - Notas médias (TMDB e IMDb): 0 a 10
  - Contagens de votos: ≥ 0
  - Popularidade: 0 a 1500 (teto para mitigar valores corrompidos)

In [0]:
# Limpa e converte métricas numéricas com formato misto.
# Estratégia: vírgula → ponto, mantém só o último ponto. Não numérico → NULL.

# Normaliza valores numéricos com formato misto e aplica faixa de negócio.
def normalize_numeric(column_name, target_type, min_val=None, max_val=None):
    col = F.trim(F.col(column_name))
    is_number_like = col.rlike(r"^[\d.,]+$")
    normalized = F.regexp_replace(col, r",", ".")
    single_dot = F.regexp_replace(normalized, r"\.(?=.*\.)", "")
    converted = single_dot.try_cast(target_type)

    conditions = is_number_like
    if min_val is not None:
        conditions = conditions & (converted >= min_val)
    if max_val is not None:
        conditions = conditions & (converted <= max_val)

    return F.when(conditions, converted)


df_bronze_movies_metrics = spark.table(f"{BRONZE_SCHEMA}.tb_movies_metrics")

# Dicionário para facilitar a tradução das colunas e eventual manutenção.
col_names_translated = {
    "id": "id_filme",
    "popularity": "popularidade",
    "vote_average": "nota_media_tmdb",
    "vote_count": "qtd_votos_tmdb",
    "averageRating": "nota_media_imdb",
    "numVotes": "qtd_votos_imdb",
    "ingestion_datetime": "ingestao_datetime",
}

# Window para deduplicação (versão mais recente por id_filme)
w = Window.partitionBy("id_filme").orderBy(
    F.col("ingestao_datetime").desc(), F.col("id_filme").asc()
)

df_silver_movies_metrics = (
    df_bronze_movies_metrics.select(list(col_names_translated.keys()))
    .withColumnsRenamed(col_names_translated)
    .withColumn("popularidade", normalize_numeric("popularidade", "decimal(18,2)", min_val=0, max_val=1500))
    .withColumn("nota_media_tmdb", normalize_numeric("nota_media_tmdb", "decimal(18,2)", min_val=0, max_val=10))
    .withColumn("qtd_votos_tmdb", normalize_numeric("qtd_votos_tmdb", "integer", min_val=0))
    .withColumn("nota_media_imdb", normalize_numeric("nota_media_imdb", "decimal(18,2)", min_val=0, max_val=10))
    .withColumn("qtd_votos_imdb", normalize_numeric("qtd_votos_imdb", "integer", min_val=0))
    .withColumn("rk", F.row_number().over(w))
    .filter(F.col("rk") == 1)
    .drop("rk", "ingestao_datetime")
)

# Validações
total = df_silver_movies_metrics.count()
unicos = df_silver_movies_metrics.select("id_filme").distinct().count()

# IDs dos filmes devem ser únicos
if total != unicos:
    print(f"{FAIL} Dedup por id_filme falhou: {total} vs {unicos}")
    raise Exception(f"Dedup por id_filme falhou: {total} linhas, {unicos} únicos.")

# Verifica se algum id_filme é nulo
nulos_id = df_silver_movies_metrics.filter(F.col("id_filme").isNull()).count()
if nulos_id > 0:
    print(f"{FAIL} {nulos_id} registros com id_filme nulo")
    raise Exception(f"id_filme nulo em {nulos_id} registros — chave natural não pode ser NULL.")

# Verifica se alguma coluna de métricas está 100% nula
metrics_cols = ["popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"]
for c in metrics_cols:
    nulos = df_silver_movies_metrics.filter(F.col(c).isNull()).count()
    if nulos == total:
        print(f"{FAIL} {c} está 100% nula. Verificar funcão.")
        raise Exception(f"Coluna {c} ficou integralmente NULL.")
    print(f"{OK} {c}: {total - nulos} não-nulos / {total}")

# Verifica nomes das colunas
expected_cols = ["id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"]
assert df_silver_movies_metrics.columns == expected_cols, (
    f"Colunas erradas: esperado {expected_cols}, "
    f"obtido {df_silver_movies_metrics.columns}"
)
print(f"{OK} Estrutura OK — 6 colunas com nomes corretos")
display(df_silver_movies_metrics.limit(20))

(df_silver_movies_metrics.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_SCHEMA}.tb_metricas_engajamento"))

print(f"{OK} Tabela gravada: {SILVER_SCHEMA}.tb_metricas_engajamento")

df_check = spark.table(f"{SILVER_SCHEMA}.tb_metricas_engajamento")
print(f"{OK} Linhas na tabela: {df_check.count()}")
df_check.printSchema()
display(df_check.limit(5))

✅ popularidade: 94836 não-nulos / 99013
✅ nota_media_tmdb: 95494 não-nulos / 99013
✅ qtd_votos_tmdb: 90629 não-nulos / 99013
✅ nota_media_imdb: 85816 não-nulos / 99013
✅ qtd_votos_imdb: 87321 não-nulos / 99013
✅ Estrutura OK — 6 colunas com nomes corretos


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1000004,1.13,0.00,0,6.80,27
1000005,0.60,0.00,0,null,40
1000007,0.60,0.00,0,4.70,10
1000011,1.17,0.00,0,4.80,16
1000014,0.60,0.00,0,7.20,15
1000030,0.62,0.00,0,null,25
1000054,0.60,0.00,0,5.90,10
1000058,1.49,6.75,6,6.20,382
1000059,0.60,0.00,0,7.70,25
1000073,13.21,6.80,15,null,236


✅ Tabela gravada: movies_medallion.silver.tb_metricas_engajamento
✅ Linhas na tabela: 99013
root
 |-- id_filme: string (nullable = true)
 |-- popularidade: decimal(18,2) (nullable = true)
 |-- nota_media_tmdb: decimal(18,2) (nullable = true)
 |-- qtd_votos_tmdb: integer (nullable = true)
 |-- nota_media_imdb: decimal(18,2) (nullable = true)
 |-- qtd_votos_imdb: integer (nullable = true)



id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1000004,1.13,0.00,0,6.80,27
1000005,0.60,0.00,0,null,40
1000007,0.60,0.00,0,4.70,10
1000011,1.17,0.00,0,4.80,16
1000014,0.60,0.00,0,7.20,15


## silver.tb_avaliacoes_usuarios

Avaliações individuais de usuários por filme.

- Validação da nota: 0-10
- Comentários vazios → "Sem comentário"
- Deduplicação por filme + usuário + nota + comentário

In [0]:
df_bronze_movies_reviews = spark.table(f"{BRONZE_SCHEMA}.tb_movies_reviews")

df_bronze_movies_reviews.printSchema()
display(df_bronze_movies_reviews.limit(50))

# Dicionário para facilitar a tradução das colunas e eventual manutenção.
col_names_translated = {
    "id": "id_filme",
    "nome": "nome_usuario",
    "nota": "nota_usuario",
    "comentario": "comentario_usuario",
}

df_silver_movies_reviews = (
    df_bronze_movies_reviews.select(list(col_names_translated.keys()))
    .withColumnsRenamed(col_names_translated)
    # Nota valida = faixa 0-10. Cast primeiro, compara depois.
    .withColumn(
        "nota_usuario",
        F.when(
            F.col("nota_usuario").try_cast("decimal(18,2)").between(0, 10),
            F.col("nota_usuario").try_cast("decimal(18,2)"),
        ),
    )
    # Comentário vazio/branco → "Sem comentário"
    .withColumn("comentario_usuario", F.trim(F.col("comentario_usuario")))
    .withColumn(
        "comentario_usuario",
        F.when(
            F.length(F.col("comentario_usuario")) > 0, F.col("comentario_usuario")
        ).otherwise("Sem comentário"),
    )
    # Remove duplicatas
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])
)

# Validações
bronze_total = df_bronze_movies_reviews.count()
total = df_silver_movies_reviews.count()

# Verifica se algum id_filme é nulo
nulos_id = df_silver_movies_reviews.filter(F.col("id_filme").isNull()).count()
if nulos_id > 0:
    print(f"{FAIL} {nulos_id} registros com id_filme nulo")
    raise Exception(f"id_filme nulo em {nulos_id} registros — chave natural não pode ser NULL.")

# Notas no silver devem estar entre 0 e 10.
notas_fora = (
    df_silver_movies_reviews
    .filter((F.col("nota_usuario") < 0) | (F.col("nota_usuario") > 10))
    .count()
)
if notas_fora > 0:
    print(f"{FAIL} {notas_fora} notas fora de 0-10 no silver.")
    raise Exception(f"{notas_fora} notas fora de 0-10 no silver.")

# Nenhum comentário vazio deve ter sobrado.
coment_vazios = (
    df_silver_movies_reviews
    .filter(
        F.col("comentario_usuario").isNull()
        | (F.length(F.col("comentario_usuario")) == 0)
    )
    .count()
)
if coment_vazios > 0:
    print(f"{FAIL} {coment_vazios} comentários vazios encontrados.")
    raise Exception(f"{coment_vazios} comentários vazios encontrados.")

print(f"Bronze total: {bronze_total}")
print(f"Silver total: {total} (removidas {bronze_total - total} duplicatas)")
print(f"{OK} Nenhuma nota fora de 0-10")
print(f"{OK} Nenhum comentário vazio")

# Estrutura: 4 colunas com nomes corretos
expected_cols = ["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"]
assert df_silver_movies_reviews.columns == expected_cols, (
    f"Colunas erradas: esperado {expected_cols}, "
    f"obtido {df_silver_movies_reviews.columns}"
)
print(f"{OK} Estrutura OK — 4 colunas com nomes corretos")

display(df_silver_movies_reviews.limit(20))

(df_silver_movies_reviews
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_SCHEMA}.tb_avaliacoes_usuarios"))

print(f"{OK} Tabela gravada: {SILVER_SCHEMA}.tb_avaliacoes_usuarios")

df_check = spark.table(f"{SILVER_SCHEMA}.tb_avaliacoes_usuarios")
print(f"{OK} Linhas na tabela: {df_check.count()}")
df_check.printSchema()
display(df_check.limit(5))

root
 |-- id: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- nota: string (nullable = true)
 |-- comentario: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id,nome,nota,comentario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,null,2026-10-16T18:46:37.408Z
637007,Lucas Reis 602,3.9,null,2026-10-16T18:46:37.408Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-10-16T18:46:37.408Z
413036,Gabriela Monteiro 401,7.5,null,2026-10-16T18:46:37.408Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-10-16T18:46:37.408Z
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo.",2026-10-16T18:46:37.408Z
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo.",2026-10-16T18:46:37.408Z
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.,2026-10-16T18:46:37.408Z
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.,2026-10-16T18:46:37.408Z
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular.",2026-10-16T18:46:37.408Z


Bronze total: 32412
Silver total: 32412 (removidas 0 duplicatas)
✅ Nenhuma nota fora de 0-10
✅ Nenhum comentário vazio
✅ Estrutura OK — 4 colunas com nomes corretos


id_filme,nome_usuario,nota_usuario,comentario_usuario
442113,Mariana Cardoso 277,4.40,Sem comentário
637007,Lucas Reis 602,3.90,Sem comentário
449479,Sérgio Freitas,0.70,Péssimo em todos os sentidos.
413036,Gabriela Monteiro 401,7.50,Sem comentário
528480,Leonardo Monteiro,6.30,Assisti até o final mas não me marcou.
387727,Maria Alves 707,8.20,"Gostei bastante, recomendo."
1032506,Amanda Castro 242,4.70,"Fraco, não recomendo."
446554,Sandra Rodrigues 736,6.50,Assisti até o final mas não me marcou.
569916,Camila Lima 489,8.20,Muito bom! Vale a pena assistir.
864552,Roberto Almeida 555,9.30,"Obra-prima do cinema, simplesmente espetacular."


✅ Tabela gravada: movies_medallion.silver.tb_avaliacoes_usuarios
✅ Linhas na tabela: 32412
root
 |-- id_filme: string (nullable = true)
 |-- nome_usuario: string (nullable = true)
 |-- nota_usuario: decimal(18,2) (nullable = true)
 |-- comentario_usuario: string (nullable = true)



id_filme,nome_usuario,nota_usuario,comentario_usuario
846452,Sérgio Freitas,6.10,Assisti até o final mas não me marcou.
493192,Felipe Ferreira 158,7.00,Filme interessante com boas atuações.
664037,Juliana Castro 595,3.90,"Fraco, não recomendo."
464493,Adriana Dias 257,0.40,Péssimo em todos os sentidos.
582666,Thiago Moreira,0.70,Não recomendo de jeito nenhum.


## silver.tb_generos
Catálogo de gêneros por filme (relação filme ↔ gênero).

- Padronização de separadores via regex
- Explode em uma linha por gênero
- Whitelist com os 19 gêneros válidos do TMDB

In [0]:
df_bronze_credits_and_tags = spark.table(f"{BRONZE_SCHEMA}.tb_credits_and_tags")

df_bronze_credits_and_tags.printSchema()
display(df_bronze_credits_and_tags.limit(50))

# Dicionário para facilitar a tradução das colunas e eventual manutenção.
col_names_translated = {
    "id": "id_filme",
    "genres": "generos",
}

# Whitelist dos 19 gêneros oficiais do TMDB
VALID_GENRES = [
    "Action", "Adventure", "Animation", "Comedy", "Crime",
    "Documentary", "Drama", "Family", "Fantasy", "History",
    "Horror", "Music", "Mystery", "Romance", "Science Fiction",
    "TV Movie", "Thriller", "War", "Western",
]

df_silver_generos = (
    df_bronze_credits_and_tags
    .select(list(col_names_translated.keys()))
    .withColumnsRenamed(col_names_translated)
    # Uniformiza separadores: qualquer caractere não alfabético ou diferente de espaço e vírgula vira vírgula
    .withColumn("generos", F.regexp_replace(F.col("generos"), r"[^a-zA-ZÀ-ÿ, ]", ","))
    .withColumn("generos", F.explode(F.split(F.col("generos"), ",")))
    .withColumn("generos", F.trim(F.col("generos")))
    .filter(F.length(F.col("generos")) > 1)
    .filter(F.col("generos").isin(*VALID_GENRES))
    .dropDuplicates(["id_filme", "generos"])
)

# Validações
total = df_silver_generos.count()

# Verifica se a tabela ficou vazia
if total == 0:
    print(f"{FAIL} Tabela de gêneros vazia")
    raise Exception("tb_generos ficou vazia — verificar pipeline de explode/filtro.")

# Verifica se algum id_filme é nulo
nulos_id = df_silver_generos.filter(F.col("id_filme").isNull()).count()
if nulos_id > 0:
    print(f"{FAIL} {nulos_id} registros com id_filme nulo")
    raise Exception(f"id_filme nulo em {nulos_id} registros — chave natural não pode ser NULL.")

# Verifica se não existem gêneros duplicados.
pares_unicos = df_silver_generos.select("id_filme", "generos").distinct().count()
if total != pares_unicos:
    print(f"{FAIL} {total - pares_unicos} duplicatas em (id_filme, generos): {total} vs {pares_unicos}")
    raise Exception(f"Duplicatas em (id_filme, generos): {total} vs {pares_unicos}")

# Verifica se todos os gêneros estão na whitelist
fora_whitelist = df_silver_generos.filter(~F.col("generos").isin(*VALID_GENRES)).count()
if fora_whitelist > 0:
    print(f"{FAIL} {fora_whitelist} gêneros fora da whitelist.")
    print(f"Gêneros permitidos: {VALID_GENRES}")
    raise Exception(f"{fora_whitelist} gêneros fora da whitelist.")

# Estrutura: 2 colunas com nomes corretos
expected_cols = ["id_filme", "generos"]
assert df_silver_generos.columns == expected_cols, (
    f"Colunas erradas: esperado {expected_cols}, "
    f"obtido {df_silver_generos.columns}"
)
print(f"{OK} Estrutura OK — 2 colunas com nomes corretos")

print(f"{OK} Total silver: {total} pares (id_filme, generos)")
print(f"{OK} Todos os gêneros na whitelist")
print(f"Filmes no bronze: {df_bronze_credits_and_tags.select('id').distinct().count()}")
print(f"Filmes no silver: {df_silver_generos.select('id_filme').distinct().count()}")

display(
    df_silver_generos.groupBy("generos")
    .count()
    .orderBy(F.col("count").desc())
    .limit(10)
)

# Persiste
(df_silver_generos
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_SCHEMA}.tb_generos"))

print(f"{OK} Tabela gravada: {SILVER_SCHEMA}.tb_generos")

df_check = spark.table(f"{SILVER_SCHEMA}.tb_generos")
print(f"{OK} Linhas na tabela: {df_check.count()}")
df_check.printSchema()
display(df_check.limit(5))

root
 |-- id: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_companies: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- spoken_languages: string (nullable = true)
 |-- keywords: string (nullable = true)
 |-- directors: string (nullable = true)
 |-- writers: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,cast,ingestion_datetime
293660,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-10-16T18:46:32.634Z
299536,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-10-16T18:46:32.634Z
299534,"Adventure, Science Fiction, Action",Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch",2026-10-16T18:46:32.634Z
475557,"Crime, Thriller, Drama","Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films","Canada, United States of America",English,"dream, street gang, society, psychopath, clown, villain, based on comic, murder, psychological thriller, criminal mastermind, mental illness, anarchy, character study, clown makeup, subway train, social realism, supervillain, tv host, 1980s, mother son relationship, origin story, falling into madness, depressing",Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson","Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais",2026-10-16T18:46:32.634Z
271110,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"Romanian, English, German, Russian","civil war, superhero, based on comic, sequel, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu), excited","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby","Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen",2026-10-16T18:46:32.634Z
284054,"Action, Adventure, Science Fiction",Marvel Studios,United States of America,"English, Korean, Swahili, Xhosa","africa, superhero, based on comic, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu)",Ryan Coogler,"Ryan Coogler, Joe Robert Cole, Stan Lee, Jack Kirby","Chadwick Boseman, Michael B. Jordan, Lupita Nyong'o, Danai Gurira, Martin Freeman, Daniel Kaluuya, Letitia Wright, Winston Duke, Sterling K. Brown, Angela Bassett",2026-10-16T18:46:32.634Z
284052,"Action, Adventure, Fantasy",Marvel Studios,UNITED STATES OF AMERICA,English,"magic, superhero, training, time, based on comic, sorcerer, doctor, neurosurgeon, wizard, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu)",Scott Derrickson,N/A,"Benedict Cumberbatch, Chiwetel Ejiofor, Rachel McAdams, Benedict Wong, Mads Mikkelsen, Tilda Swinton, Michael Stuhlbarg, Benjamin Bratt, Scott Adkins, Zara Phythian",2026-10-16T18:46:32.634Z
315635,"Action, Adventure, Science Fiction, Drama","Ma

✅ Estrutura OK — 2 colunas com nomes corretos
✅ Total silver: 142188 pares (id_filme, generos)
✅ Todos os gêneros na whitelist
Filmes no bronze: 99006
Filmes no silver: 85155


generos,count
Drama,32687
Documentary,19255
Comedy,18865
Thriller,10415
Horror,9871
Romance,7724
Action,6114
Crime,4795
Animation,4532
TV Movie,4122


✅ Tabela gravada: movies_medallion.silver.tb_generos
✅ Linhas na tabela: 142188
root
 |-- id_filme: string (nullable = true)
 |-- generos: string (nullable = true)



id_filme,generos
329865,Mystery
269149,Animation
429617,Science Fiction
330459,Action
127380,Family


## silver.tb_pessoas_empresas
Dimensão unificada de entidades categorizada por tipo.

- Explode de cada coluna em linhas individuais
- Padronização de capitalização (`initcap`) e `trim`
- Categorização: Ator, Diretor, Roteirista, Produtora
- União das 4 fontes e deduplicação

In [0]:
df_bronze_credits_and_tags = spark.table(f"{BRONZE_SCHEMA}.tb_credits_and_tags")

# Função para explodir uma coluna de entidades em linhas individuais com tipo especificado
def explode_entity(df, source_col, entity_type):
    return (
        df
        .select(
            F.col("id").alias("id_filme"),
            F.explode(F.split(F.col(source_col), ",")).alias("nome"),
        )
        .withColumn("nome", F.initcap(F.trim(F.col("nome"))))
        .withColumn("tipo", F.lit(entity_type))
        .filter(F.length(F.col("nome")) > 1)
    )

df_cast = explode_entity(df_bronze_credits_and_tags, "cast", "Ator")
if df_cast.count() == 0:
    print(f"{FAIL} Tabela df_cast vazia — verificar função de explode/filtro.")
    raise Exception(f"Tabela df_cast vazia — verificar função de explode/filtro.")

df_directors = explode_entity(df_bronze_credits_and_tags, "directors", "Diretor")
if df_directors.count() == 0:
    print(f"{FAIL} Tabela df_directors vazia — verificar função de explode/filtro.")
    raise Exception(f"Tabela df_directors vazia — verificar função de explode/filtro.")

df_writers = explode_entity(df_bronze_credits_and_tags, "writers", "Roteirista")
if df_writers.count() == 0:
    print(f"{FAIL} Tabela df_writers vazia — verificar função de explode/filtro.")
    raise Exception(f"Tabela df_writers vazia — verificar função de explode/filtro.")

df_production_companies = explode_entity(df_bronze_credits_and_tags, "production_companies", "Produtora")
if df_production_companies.count() == 0:
    print(f"{FAIL} Tabela df_production_companies vazia — verificar função de explode/filtro.")
    raise Exception(f"Tabela df_production_companies vazia — verificar função de explode/filtro.")

# União + filtros de ruído (column shift e nomes absurdos)
df_silver_pessoas_empresas = (
    df_cast
    .unionByName(df_directors)
    .unionByName(df_writers)
    .unionByName(df_production_companies)
    .dropDuplicates(["id_filme", "nome", "tipo"])
    # Filtra para nomes com pelo menos um caractere legítimo (letras, dígitos, espaços,
    # ponto, hífen, apóstrofo, &)e no máximo 9 palavras (evitar column shift da sinopse, por exemplo)
    .filter(F.col("nome").rlike(r"^[A-Za-z0-9À-ÿ\s\.\-'&]+$"))
    .filter(F.size(F.split(F.col("nome"), r"\s+")) < 10)
)

# Validações
total = df_silver_pessoas_empresas.count()

# Verifica se a tabela ficou vazia
if total == 0:
    print(f"{FAIL} Tabela df_silver_pessoas_empresas vazia — verificar union e filtro.")
    raise Exception(f"Tabela df_silver_pessoas_empresas vazia — verificar union e filtro.")

# Verifica se algum id_filme é nulo
nulos_id = df_silver_pessoas_empresas.filter(F.col("id_filme").isNull()).count()
if nulos_id > 0:
    print(f"{FAIL} {nulos_id} registros com id_filme nulo")
    raise Exception(f"id_filme nulo em {nulos_id} registros — chave natural não pode ser NULL.")

# Verifica se há duplicatas
pares_unicos = df_silver_pessoas_empresas.select("id_filme", "nome", "tipo").distinct().count()
if total != pares_unicos:
    print(f"{FAIL} Duplicatas em (id_filme, nome, tipo): {total} vs {pares_unicos}")
    raise Exception(f"Duplicatas em (id_filme, nome, tipo): {total} vs {pares_unicos}")

# Verificar se os tipos estão corretos
tipos_validos = {"Ator", "Diretor", "Roteirista", "Produtora"}
tipos_encontrados = {
    row["tipo"] for row in df_silver_pessoas_empresas.select("tipo").distinct().collect()
}
if tipos_encontrados != tipos_validos:
    print(f"{FAIL} Tipos inesperados: {tipos_encontrados - tipos_validos}")
    raise Exception(f"Tipos inesperados: {tipos_encontrados - tipos_validos}")

# Estrutura: 3 colunas com nomes corretos
expected_cols = ["id_filme", "nome", "tipo"]
assert df_silver_pessoas_empresas.columns == expected_cols, (
    f"Colunas erradas: esperado {expected_cols}, "
    f"obtido {df_silver_pessoas_empresas.columns}"
)
print(f"{OK} Estrutura OK — 3 colunas com nomes corretos")

print(f"{OK} Total: {total} pares (id_filme, nome, tipo)")
print(f"{OK} Tipos válidos: {tipos_encontrados}")

# Visualização
display(
    df_silver_pessoas_empresas.groupBy("tipo")
    .count()
    .orderBy(F.col("count").desc())
)

df_silver_pessoas_empresas.printSchema()

# Persiste 

(df_silver_pessoas_empresas
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_SCHEMA}.tb_pessoas_empresas"))

print(f"{OK} Tabela gravada: {SILVER_SCHEMA}.tb_pessoas_empresas")

df_check = spark.table(f"{SILVER_SCHEMA}.tb_pessoas_empresas")
print(f"{OK} Linhas na tabela: {df_check.count()}")
df_check.printSchema()
display(df_check.limit(5))

✅ Estrutura OK — 3 colunas com nomes corretos
✅ Total: 878142 pares (id_filme, nome, tipo)
✅ Tipos válidos: {'Produtora', 'Ator', 'Diretor', 'Roteirista'}


tipo,count
Ator,533858
Roteirista,126638
Produtora,115634
Diretor,102012


root
 |-- id_filme: string (nullable = true)
 |-- nome: string (nullable = false)
 |-- tipo: string (nullable = false)

✅ Tabela gravada: movies_medallion.silver.tb_pessoas_empresas
✅ Linhas na tabela: 878142
root
 |-- id_filme: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- tipo: string (nullable = true)



id_filme,nome,tipo
299536,Don Cheadle,Ator
263115,Hugh Jackman,Ator
354912,Anthony Gonzalez,Ator
419430,Daniel Kaluuya,Ator
424694,Mike Myers,Ator


## silver.tb_cotacao_dolar

Série temporal contínua da cotação USD/BRL.

- Conversão de data para `date` (remove hora)
- Geração de calendário diário entre data mínima e máxima
- Forward fill via `last(ignorenulls=True)` para fins de semana e feriados


In [0]:
df_bronze_cotacao_dolar = spark.table(f"{BRONZE_SCHEMA}.tb_cotacao_dolar")

# Dicionário para facilitar a tradução das colunas e eventual manutenção.
col_names_translated = {
    "cotacaoCompra": "cotacao_compra",
    "dataHoraCotacao": "data_cotacao",
}

# Renomeia colunas e converte data para DATE (remove hora)
df_cotacao_base = (
    df_bronze_cotacao_dolar
    .select(list(col_names_translated.keys()))
    .withColumnsRenamed(col_names_translated)
    .withColumn("data_cotacao", F.to_date(F.col("data_cotacao")))
)

# Gera calendário diário entre min e max (cobre fins de semana/feriados)
date_range = df_cotacao_base.agg(
    F.min("data_cotacao").alias("min_date"),
    F.max("data_cotacao").alias("max_date"),
).first()

if date_range is None or date_range["min_date"] is None:
    print(f"{FAIL} Tabela bronze de cotação vazia — impossível gerar calendário")
    raise Exception("tb_cotacao_dolar (bronze) vazia ou sem datas válidas.")

df_calendar = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(date_range["min_date"]),
                F.lit(date_range["max_date"]),
                F.expr("interval 1 day"),
            )
        ).alias("data_cotacao")
    )
)

# Forward fill (fins de semana/feriados herdam a última cotação)
w = Window.orderBy("data_cotacao").rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)

df_silver_cotacao_dolar = (
    df_calendar
    .join(df_cotacao_base, on="data_cotacao", how="left")
    .withColumn("cotacao_compra", F.last("cotacao_compra", ignorenulls=True).over(w))
)

# Validações
total_dias = df_silver_cotacao_dolar.count()

# Verifica se há dias sem cotação (vazios)
nulos = df_silver_cotacao_dolar.filter(F.col("cotacao_compra").isNull()).count()
if nulos > 0:
    print(f"{FAIL} {nulos} dias sem cotação após forward fill.")
    raise Exception(f"{nulos} dias sem cotação após forward fill.")

# Verifica se a série é contínua
dias_esperados = (date_range["max_date"] - date_range["min_date"]).days + 1
if total_dias != dias_esperados:
    print(f"{FAIL} Série incompleta: {total_dias} dias vs {dias_esperados} esperados.")
    raise Exception(f"Série incompleta: {total_dias} dias vs {dias_esperados} esperados.")

print(f"{OK} Série contínua: {total_dias} dias ({date_range['min_date']} a {date_range['max_date']})")
print(f"{OK} Nenhum dia sem cotação")

display(df_silver_cotacao_dolar.orderBy("data_cotacao"))

# Persiste
(df_silver_cotacao_dolar
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_SCHEMA}.tb_cotacao_dolar"))

print(f"{OK} Tabela gravada: {SILVER_SCHEMA}.tb_cotacao_dolar")

df_check = spark.table(f"{SILVER_SCHEMA}.tb_cotacao_dolar")
print(f"{OK} Linhas na tabela: {df_check.count()}")
df_check.printSchema()
display(df_check.limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ Série contínua: 8 dias (2026-09-29 a 2026-10-06)
✅ Nenhum dia sem cotação


data_cotacao,cotacao_compra
2026-09-29,5.2198
2026-09-30,5.1803
2026-10-01,5.2073
2026-10-02,5.2232
2026-10-03,5.2232
2026-10-04,5.2232
2026-10-05,4.9853
2026-10-06,4.9692


✅ Tabela gravada: movies_medallion.silver.tb_cotacao_dolar
✅ Linhas na tabela: 8
root
 |-- data_cotacao: date (nullable = true)
 |-- cotacao_compra: double (nullable = true)



data_cotacao,cotacao_compra
2026-09-29,5.2198
2026-09-30,5.1803
2026-10-01,5.2073
2026-10-02,5.2232
2026-10-03,5.2232


## Resumo Final — Camada Silver

Apenas reporta. Validações por bloco já deram raise antes.

In [0]:
# Resumo final: apenas reporta. Validações por bloco já deram raise antes.

tables = [
    "tb_info_filmes",
    "tb_financeiro_filmes",
    "tb_metricas_engajamento",
    "tb_avaliacoes_usuarios",
    "tb_generos",
    "tb_pessoas_empresas",
    "tb_cotacao_dolar",
]


print("=" * 60)
print("RESUMO DA CAMADA SILVER")
print("=" * 60)

print(f"Tabelas em {SILVER_SCHEMA}:\n")
display(spark.sql(f"SHOW TABLES IN {SILVER_SCHEMA}"))

for t in tables:
    full = f"{SILVER_SCHEMA}.{t}"
    n = spark.table(full).count()
    print(f"{OK} {full}: {n} linhas")

print(f"Executado em: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

RESUMO DA CAMADA SILVER
Tabelas em movies_medallion.silver:



database,tableName,isTemporary
silver,tb_avaliacoes_usuarios,false
silver,tb_cotacao_dolar,false
silver,tb_financeiro_filmes,false
silver,tb_generos,false
silver,tb_info_filmes,false
silver,tb_metricas_engajamento,false
silver,tb_pessoas_empresas,false


✅ movies_medallion.silver.tb_info_filmes: 97879 linhas
✅ movies_medallion.silver.tb_financeiro_filmes: 99006 linhas
✅ movies_medallion.silver.tb_metricas_engajamento: 99013 linhas
✅ movies_medallion.silver.tb_avaliacoes_usuarios: 32412 linhas
✅ movies_medallion.silver.tb_generos: 142188 linhas
✅ movies_medallion.silver.tb_pessoas_empresas: 878142 linhas
✅ movies_medallion.silver.tb_cotacao_dolar: 8 linhas
Executado em: 2026-10-16 20:00:27
